In [1]:
import pandas as pd
import numpy as np

TARGETS = ["y"]

In [2]:
results_1l = pd.read_excel("resultados-1l-v2.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
#     [results_1l, results_2l,],
#     ignore_index=True )
results = results_1l


In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZx1_y,MSE_ZZx1_y,R2_ZZx2_y,MSE_ZZx2_y,R2_ZZxReto_y,MSE_ZZxReto_y
0,model_arch1_r0.01_Ld0.5_Lp0.5_seed9974,[1],0.5,0.5,0.01,9974,0.994327,0.942682,0.848566,0.806073,-4.036241,0.129933
1,model_arch1_r0.01_Ld0.5_Lp0.5_seed6686,[1],0.5,0.5,0.01,6686,0.995140,0.942600,0.784523,0.796105,-4.453217,0.079322
2,model_arch1_r0.01_Ld0.5_Lp0.5_seed8467,[1],0.5,0.5,0.01,8467,0.976474,0.893269,0.652937,0.621435,-0.033474,0.365163
3,model_arch1_r0.01_Ld0.5_Lp0.5_seed2617,[1],0.5,0.5,0.01,2617,0.994018,0.942436,0.869441,0.809846,-3.914866,0.146542
4,model_arch1_r0.01_Ld0.5_Lp0.5_seed5011,[1],0.5,0.5,0.01,5011,0.989804,0.868870,0.557989,0.789269,0.785245,0.476912
...,...,...,...,...,...,...,...,...,...,...,...,...
1469,model_arch49_r0.01_Ld0.3_Lp0.7_seed2376,[49],0.3,0.7,0.01,2376,0.995411,0.969868,0.869534,0.870483,-0.001673,0.670949
1470,model_arch49_r0.01_Ld0.3_Lp0.7_seed1788,[49],0.3,0.7,0.01,1788,0.992977,0.953252,0.698904,0.818223,-0.799937,0.593411
1471,model_arch49_r0.9_Ld0.3_Lp0.7_seed2494,[49],0.3,0.7,0.90,2494,0.994684,0.965616,0.845404,0.854004,-0.310436,0.635844
1472,model_arch49_r0.9_Ld0.3_Lp0.7_seed2917,[49],0.3,0.7,0.90,2917,0.995226,0.965859,0.857424,0.853885,-0.002732,0.644159


In [6]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZx1":     "Train",
    "ZZx2":     "Val",
    "ZZxReto":  "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 5  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33


for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"] - 
        0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 5 MODELOS - y


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
481,model_arch17_r0.01_Ld0.5_Lp0.5_seed6686,[17],0.999419,0.991183,0.922728,0.957189
609,model_arch21_r0.9_Ld0.5_Lp0.5_seed5011,[21],0.995767,0.987182,0.846694,0.925412
1285,model_arch43_r0.9_Ld0.7_Lp0.3_seed9974,[43],0.998170,0.960302,0.854671,0.920900
1280,model_arch43_r0.01_Ld0.7_Lp0.3_seed9974,[43],0.999044,0.943176,0.857036,0.916601
115,model_arch4_r0.9_Ld0.7_Lp0.3_seed9974,[4],0.998887,0.964123,0.825890,0.911186



📊 MÉTRICAS COMPLETAS - TOP 5 (y)


,model,Neurons,R2_ZZx1_y,R2_ZZx2_y,R2_ZZxReto_y,R2_train_mean,R2_val_mean,R2_test_mean,Score
481,model_arch17_r0.01_Ld0.5_Lp0.5_seed6686,[17],0.999419,0.991183,0.922728,0.999419,0.991183,0.922728,0.957189
609,model_arch21_r0.9_Ld0.5_Lp0.5_seed5011,[21],0.995767,0.987182,0.846694,0.995767,0.987182,0.846694,0.925412
1285,model_arch43_r0.9_Ld0.7_Lp0.3_seed9974,[43],0.998170,0.960302,0.854671,0.998170,0.960302,0.854671,0.920900
1280,model_arch43_r0.01_Ld0.7_Lp0.3_seed9974,[43],0.999044,0.943176,0.857036,0.999044,0.943176,0.857036,0.916601
115,model_arch4_r0.9_Ld0.7_Lp0.3_seed9974,[4],0.998887,0.964123,0.825890,0.998887,0.964123,0.825890,0.911186


In [7]:
final_table.to_excel("BestModels-otm.xlsx")